## Baselines

Baseline 1 — Persistência (naive)

Nesta célula vamos avaliar o baseline mais simples: prever o valor do mês t como sendo o valor observado no mês anterior. Em outras palavras, a previsão é:

ŷ(t) = y(t-1)

Esse baseline define um piso mínimo e serve para verificar se qualquer modelo mais complexo realmente agrega valor em relação a “copiar o último valor”. A avaliação será feita com walk-forward (janela expansível): em cada mês t, usamos os dados até t-1 e avaliamos a previsão em t.

In [7]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.arima.model import ARIMA
import warnings

warnings.filterwarnings("ignore")


In [4]:
# Baseline 1: Persistência (ŷ(t) = y(t-1)) com walk-forward expanding window

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

TARGET = "inadimplencia_pf_total"
LAG_COL = "y_lag1"

MIN_TRAIN_MONTHS = 48  # 4 anos (ajuste para 36–48 se quiser)
test_dates = df_model.index[MIN_TRAIN_MONTHS:]

y_true = df_model.loc[test_dates, TARGET].astype(float)
y_pred = df_model.loc[test_dates, LAG_COL].astype(float)  # persistência

def rmse(a, b):
    return np.sqrt(mean_squared_error(a, b))

metrics_persist = {
    "MAE": mean_absolute_error(y_true, y_pred),
    "RMSE": rmse(y_true, y_pred),
    "N": len(y_true),
    "Período início": y_true.index.min().date(),
    "Período fim": y_true.index.max().date(),
}

metrics_persist

{'MAE': 0.08142857142857143,
 'RMSE': 0.10600539069850579,
 'N': 105,
 'Período início': datetime.date(2017, 4, 1),
 'Período fim': datetime.date(2025, 12, 1)}

Baseline 2 — AR(1) linear (regressão com y(t-1))

Vamos avaliar um baseline univariado um pouco mais flexível do que a persistência. Em vez de assumir que y(t) é exatamente igual a y(t-1), ajustamos uma regressão linear usando apenas o lag 1 do alvo:

y(t) = a + b · y(t-1)

A avaliação segue o mesmo esquema walk-forward (janela expansível): para cada mês t, treinamos o modelo com os dados até t-1 e prevemos t. Assim, comparamos diretamente este baseline com a persistência usando as mesmas datas e métricas.

In [6]:
# Baseline 2 AR(1) linear

X_COL = [LAG_COL]

y_true_list = []
y_pred_list = []
dates = []

for t in test_dates:
    train = df_model.loc[:t].iloc[:-1]  # até t-1
    test = df_model.loc[[t]]            # t

    lr = LinearRegression()
    lr.fit(train[X_COL], train[TARGET])

    pred = float(lr.predict(test[X_COL])[0])

    y_true_list.append(float(test[TARGET].iloc[0]))
    y_pred_list.append(pred)
    dates.append(t)

results_ar1 = pd.DataFrame(
    {"y_true": y_true_list, "y_pred": y_pred_list},
    index=pd.to_datetime(dates),
)

def rmse(a, b):
    return np.sqrt(mean_squared_error(a, b))

metrics_ar1 = {
    "MAE": mean_absolute_error(results_ar1["y_true"], results_ar1["y_pred"]),
    "RMSE": rmse(results_ar1["y_true"], results_ar1["y_pred"]),
    "N": len(results_ar1),
    "Período início": results_ar1.index.min().date(),
    "Período fim": results_ar1.index.max().date(),
}

metrics_ar1

{'MAE': 0.08553620452047377,
 'RMSE': 0.11113887528607662,
 'N': 105,
 'Período início': datetime.date(2017, 4, 1),
 'Período fim': datetime.date(2025, 12, 1)}

Baseline 3 — ARIMA(1,1,0) univariado

Baseline estatístico clássico para séries temporais: ARIMA(1,1,0), usando apenas o histórico do alvo. A ideia é modelar a dinâmica da série (incluindo a possibilidade de não estacionaridade via diferenciação) e prever um passo à frente. A avaliação seguirá o mesmo esquema walk-forward (janela expansível): para cada mês t, ajustamos o ARIMA usando os dados até t-1 e prevemos y(t). Em seguida, comparamos MAE/RMSE com os baselines anteriores.

In [8]:
# Baseline 3: ARIMA(1,1,0) univariado com walk-forward expanding window

y_true_list = []
y_pred_list = []
dates = []

# Série alvo completa no df_model
y_series = df_model[TARGET].astype(float)

for t in test_dates:
    train_y = y_series.loc[:t].iloc[:-1]  # até t-1
    test_y = y_series.loc[t]              # y(t)

    # ARIMA(1,1,0) treinado só no histórico do alvo
    model = ARIMA(train_y, order=(1, 1, 0))
    fit = model.fit()

    pred = float(fit.forecast(steps=1).iloc[0])

    y_true_list.append(float(test_y))
    y_pred_list.append(pred)
    dates.append(t)

results_arima110 = pd.DataFrame(
    {"y_true": y_true_list, "y_pred": y_pred_list},
    index=pd.to_datetime(dates),
)

def rmse(a, b):
    return np.sqrt(mean_squared_error(a, b))

metrics_arima110 = {
    "MAE": mean_absolute_error(results_arima110["y_true"], results_arima110["y_pred"]),
    "RMSE": rmse(results_arima110["y_true"], results_arima110["y_pred"]),
    "N": len(results_arima110),
    "Período início": results_arima110.index.min().date(),
    "Período fim": results_arima110.index.max().date(),
}

metrics_arima110

{'MAE': 0.07793546401410789,
 'RMSE': 0.10312988344730709,
 'N': 105,
 'Período início': datetime.date(2017, 4, 1),
 'Período fim': datetime.date(2025, 12, 1)}

Com base no backtesting walk-forward (janela expansível) no período de 2017-04 a 2025-12 (105 previsões), observamos que:

- O baseline de persistência (prever y(t) = y(t-1)) já apresenta bom desempenho, com MAE de 0,0814 e RMSE de 0,1060, indicando forte inércia no target.

- O baseline AR(1) linear (regressão usando apenas y(t-1)) teve desempenho inferior à persistência (MAE 0,0855; RMSE 0,1111), sugerindo que o ajuste mensal do coeficiente adiciona ruído e não melhora a previsão em relação ao naive.

- O ARIMA(1,1,0) univariado foi o melhor baseline entre os testados, superando a persistência (MAE 0,0779; RMSE 0,1031). Assim, ele se torna o principal benchmark a ser batido pelos modelos com variáveis exógenas.

A partir daqui, a hipótese central passa a ser: variáveis macroeconômicas e financeiras só serão consideradas úteis se conseguirem melhorar de forma consistente o desempenho fora da amostra em relação ao ARIMA(1,1,0) univariado.

# Modelos com Variáveis Exógenas

## SARIMAX/ARIMAX — ARIMA(1,1,0) com variáveis exógenas

Vamos avaliar um modelo com variáveis exógenas (ARIMAX/SARIMAX) usando o mesmo protocolo walk-forward (janela expansível). A ideia é verificar se as features macro/financeiras (já defasadas no df_model, portanto sem vazamento) trazem ganho preditivo fora da amostra em relação ao baseline ARIMA(1,1,0) univariado. Manteremos a ordem (1,1,0) para comparabilidade direta.

In [9]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings

warnings.filterwarnings("ignore")

In [10]:
# SARIMAX: ARIMA(1,1,0) + exógenas com walk-forward expanding window

# X exógenas: todas as colunas exceto alvo
X_cols = [c for c in df_model.columns if c != TARGET]

y_true_list = []
y_pred_list = []
dates = []

y_series = df_model[TARGET].astype(float)

for t in test_dates:
    train = df_model.loc[:t].iloc[:-1]   # até t-1
    test  = df_model.loc[[t]]            # t

    train_y = train[TARGET].astype(float)
    train_X = train[X_cols].astype(float)

    test_X = test[X_cols].astype(float)

    # SARIMAX com order (1,1,0)
    model = SARIMAX(
        train_y,
        exog=train_X,
        order=(1, 1, 0),
        trend="c",                 # intercepto (pode ajustar depois)
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    fit = model.fit(disp=False)

    pred = float(fit.forecast(steps=1, exog=test_X).iloc[0])

    y_true_list.append(float(test[TARGET].iloc[0]))
    y_pred_list.append(pred)
    dates.append(t)

results_sarimax = pd.DataFrame(
    {"y_true": y_true_list, "y_pred": y_pred_list},
    index=pd.to_datetime(dates),
)

def rmse(a, b):
    return np.sqrt(mean_squared_error(a, b))

metrics_sarimax = {
    "MAE": mean_absolute_error(results_sarimax["y_true"], results_sarimax["y_pred"]),
    "RMSE": rmse(results_sarimax["y_true"], results_sarimax["y_pred"]),
    "N": len(results_sarimax),
    "Período início": results_sarimax.index.min().date(),
    "Período fim": results_sarimax.index.max().date(),
}

metrics_sarimax

{'MAE': 0.08012318670696814,
 'RMSE': 0.10571984220881252,
 'N': 105,
 'Período início': datetime.date(2017, 4, 1),
 'Período fim': datetime.date(2025, 12, 1)}

Mini-tuning do SARIMAX (walk-forward)

Nesta etapa vamos testar pequenas variações do SARIMAX para verificar se a piora observada com exógenas foi causada por especificação inadequada. Vamos manter o mesmo protocolo walk-forward (janela expansível) e comparar combinações simples de: (i) ordem ARIMA (p,d,q), (ii) uso ou não de intercepto (trend) e (iii) subconjuntos parcimoniosos de variáveis exógenas. O objetivo não é fazer um grid grande, mas sim checar ajustes de baixo custo que podem estabilizar o modelo e, se possível, superar o ARIMA(1,1,0) univariado.

In [11]:
# Mini-tuning SARIMAX com walk-forward expanding window

def rmse(a, b):
    return np.sqrt(mean_squared_error(a, b))

# 1) Definir conjuntos de exógenas (parsimoniosos -> completo)
EXOG_SETS = {
    "selic+juros": ["selic_lag8", "juros_pf_lag6"],
    "selic+juros+ibc": ["selic_lag8", "juros_pf_lag6", "ibc_lag12"],
    "all_exog": [c for c in df_model.columns if c not in [TARGET]],
}

# 2) Pequeno grid de ordens e trend
ORDERS = [(1,1,0), (0,1,1), (1,1,1)]
TRENDS = ["n", "c"]  # sem intercepto vs com intercepto

def walkforward_sarimax(order, trend, exog_cols, min_train_months=MIN_TRAIN_MONTHS):
    test_dates_local = df_model.index[min_train_months:]
    y_true, y_pred = [], []

    for t in test_dates_local:
        train = df_model.loc[:t].iloc[:-1]
        test  = df_model.loc[[t]]

        train_y = train[TARGET].astype(float)
        train_X = train[exog_cols].astype(float)
        test_X  = test[exog_cols].astype(float)

        model = SARIMAX(
            train_y,
            exog=train_X,
            order=order,
            trend=trend,
            enforce_stationarity=False,
            enforce_invertibility=False,
        )
        fit = model.fit(disp=False)

        pred = float(fit.forecast(steps=1, exog=test_X).iloc[0])
        y_true.append(float(test[TARGET].iloc[0]))
        y_pred.append(pred)

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    return {
        "order": str(order),
        "trend": trend,
        "exog_set": None,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "N": len(y_true),
    }

# 3) Rodar grid
rows = []
for exog_name, exog_cols in EXOG_SETS.items():
    for order in ORDERS:
        for trend in TRENDS:
            try:
                out = walkforward_sarimax(order=order, trend=trend, exog_cols=exog_cols)
                out["exog_set"] = exog_name
                rows.append(out)
            except Exception as e:
                rows.append({
                    "order": str(order),
                    "trend": trend,
                    "exog_set": exog_name,
                    "MAE": np.nan,
                    "RMSE": np.nan,
                    "N": 0,
                    "error": str(e)[:200],
                })

results_grid = pd.DataFrame(rows).sort_values(["MAE", "RMSE"])
display(results_grid)

# Mostrar o top-5
display(results_grid.head(5))

,order,trend,exog_set,MAE,RMSE,N
4,"(1, 1, 1)",n,selic+juros,0.075088,0.101154,105
5,"(1, 1, 1)",c,selic+juros,0.075377,0.101124,105
10,"(1, 1, 1)",n,selic+juros+ibc,0.076202,0.102033,105
2,"(0, 1, 1)",n,selic+juros,0.076357,0.102449,105
0,"(1, 1, 0)",n,selic+juros,0.076475,0.103069,105
11,"(1, 1, 1)",c,selic+juros+ibc,0.077308,0.102658,105
6,"(1, 1, 0)",n,selic+juros+ibc,0.077431,0.103870,105
8,"(0, 1, 1)",n,selic+juros+ibc,0.077615,0.103451,105
3,"(0, 1, 1)",c,selic+juros,0.078604,0.103566,105
14,"(0, 1, 1)",n,all_exog,0.078629,0.102462,105


,order,trend,exog_set,MAE,RMSE,N
4,"(1, 1, 1)",n,selic+juros,0.075088,0.101154,105
5,"(1, 1, 1)",c,selic+juros,0.075377,0.101124,105
10,"(1, 1, 1)",n,selic+juros+ibc,0.076202,0.102033,105
2,"(0, 1, 1)",n,selic+juros,0.076357,0.102449,105
0,"(1, 1, 0)",n,selic+juros,0.076475,0.103069,105


# 02 — Modeling (old)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.features import load_panel_and_build_features
from src.models import get_all_models

sns.set_style("whitegrid")

In [ ]:
X, y = load_panel_and_build_features()
print(f"Features: {X.shape[1]}")
print(f"Observations: {y.notna().sum()}")

## Single Train/Test Split (Quick Exploration)

In [ ]:
# Use ~80% for training
valid_mask = y.notna()
valid_dates = y[valid_mask].index
split_idx = int(len(valid_dates) * 0.8)

train_dates = valid_dates[:split_idx]
test_dates = valid_dates[split_idx:]

X_train, y_train = X.loc[train_dates], y.loc[train_dates]
X_test, y_test = X.loc[test_dates], y.loc[test_dates]

print(f"Train: {len(train_dates)} obs ({train_dates[0].date()} to {train_dates[-1].date()})")
print(f"Test:  {len(test_dates)} obs ({test_dates[0].date()} to {test_dates[-1].date()})")

In [ ]:
models = get_all_models()
results = {}

for model in models:
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[model.name] = preds
    mae = np.mean(np.abs(y_test.values - preds))
    print(f"{model.name}: MAE = {mae:.4f}")

## Residual Analysis

In [ ]:
fig, axes = plt.subplots(len(models), 1, figsize=(14, 3 * len(models)))
for ax, model in zip(axes, models):
    residuals = y_test.values - results[model.name]
    ax.plot(test_dates, residuals, "o-", markersize=3)
    ax.axhline(0, color="k", linestyle="--", alpha=0.5)
    ax.set_ylabel(f"{model.name}\nResiduals")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()